# barbac vs Shepherd benchmark

Runs both clustering methods on the same input, scores them against the ground truth using the Johnson et al. 2023 metrics:

- **Pearson R** on log10 counts of matched centroids
- **FN** rate — true barcodes recovered as no centroid
- **FP** rate — centroids that aren't true barcodes
- **WS** rate — FP centroids within `MAX_DIST` Levenshtein of any true barcode (wrong-sequence calls)

## Prerequisites

1. Run `benchmark/export_barbac_result.R` from R first — it writes `barbac_result.csv` to the benchmark folder.
2. Shepherd must be cloned at `~/Documents/Projects/Barcodes/barbac-benchmark/tools/Shepherd/`. The notebook reuses the cached `shepherd_input_pb_freq.csv` if present, otherwise re-runs Shepherd (~2–3 min).

In [ ]:
import sys, subprocess
try:
    import rapidfuzz  # noqa
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rapidfuzz"])

from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rapidfuzz.distance import Levenshtein
from rapidfuzz import process

In [ ]:
WORK = Path("~/Documents/Projects/Barcodes/barbac-benchmark").expanduser()
SHEPHERD_DIR = WORK / "tools" / "Shepherd"

INPUT_CSV      = WORK / "barbac_benchmark_input.csv"
TRUE_CSV       = WORK / "true_counts.csv"
SHEPHERD_INPUT = WORK / "shepherd_input.txt"
SHEPHERD_OUT   = WORK / "shepherd_input_pb_freq.csv"
BARBAC_RESULT  = WORK / "barbac_result.csv"

MAX_DIST   = 3
BARCODE_LEN = 20

for p in [INPUT_CSV, TRUE_CSV, SHEPHERD_DIR]:
    assert p.exists(), f"Missing: {p}"

## 1. Load ground truth + input

In [ ]:
true_counts = pd.read_csv(TRUE_CSV)
true_counts.columns = ["barcode", "true_count"]

input_data = pd.read_csv(INPUT_CSV)

print(f"True barcodes: {len(true_counts):,}")
print(f"Input rows:    {len(input_data):,}")
print(f"Total reads:   {input_data['counts'].sum():,}")
true_counts.head()

## 2. Run Shepherd (or use cached output)

In [ ]:
if not SHEPHERD_INPUT.exists():
    print("Writing Shepherd input...")
    input_data[["barcode", "counts"]].to_csv(
        SHEPHERD_INPUT, sep="\t", header=False, index=False
    )

if SHEPHERD_OUT.exists():
    print(f"Using cached Shepherd output: {SHEPHERD_OUT.name}")
else:
    cmd = [
        sys.executable, str(SHEPHERD_DIR / "shepherd_t0.py"),
        "-f", str(SHEPHERD_INPUT),
        "-l", str(BARCODE_LEN),
        "-eps", str(MAX_DIST),
    ]
    print("Running:", " ".join(cmd))
    t0 = time.time()
    subprocess.run(cmd, cwd=WORK, check=True)
    print(f"Shepherd finished in {(time.time()-t0)/60:.1f} min")

## 3. Load both result sets

Each method's result is reduced to two columns: `central_barcode` and `sum_counts`.

In [ ]:
shepherd = pd.read_csv(SHEPHERD_OUT)
shepherd.columns = ["central_barcode", "sum_counts"]

if not BARBAC_RESULT.exists():
    raise FileNotFoundError(
        f"Missing {BARBAC_RESULT.name}. Run benchmark/export_barbac_result.R from R first."
    )
barbac = pd.read_csv(BARBAC_RESULT)

print(f"Shepherd centroids: {len(shepherd):,}")
print(f"barbac centroids:   {len(barbac):,}")
print(f"Ground truth:       {len(true_counts):,}")

## 4. Evaluation function

Identical metrics to `Untitled.R::evaluate_clustering()`. WS uses batched `rapidfuzz` Levenshtein with `score_cutoff = MAX_DIST + 1` so each FP only needs to find its nearest true barcode.

In [ ]:
def evaluate(result: pd.DataFrame, true_counts: pd.DataFrame, max_dist: int = MAX_DIST,
             batch: int = 200) -> dict:
    true_set  = set(true_counts["barcode"])
    true_list = true_counts["barcode"].tolist()
    n_true    = len(true_set)

    centroids    = result["central_barcode"].tolist()
    centroid_set = set(centroids)

    fn = list(true_set - centroid_set)
    fp = list(centroid_set - true_set)

    matched = result.merge(
        true_counts, left_on="central_barcode", right_on="barcode", how="inner"
    )
    pearson_r = float(np.corrcoef(
        np.log10(matched["sum_counts"]),
        np.log10(matched["true_count"]),
    )[0, 1])

    # WS: FP centroids whose nearest true barcode is within max_dist.
    # With score_cutoff=max_dist, rapidfuzz returns cutoff+1 for any pair
    # whose true distance exceeds max_dist, so min<=max_dist is the test.
    ws = 0
    if fp:
        for i in range(0, len(fp), batch):
            chunk = fp[i : i + batch]
            dm = process.cdist(
                chunk, true_list,
                scorer=Levenshtein.distance,
                score_cutoff=max_dist,
                dtype=np.uint8,
            )
            min_d = dm.min(axis=1)
            ws += int((min_d <= max_dist).sum())

    return {
        "n_centroids": len(centroids),
        "pearson_r":   pearson_r,
        "fn":          len(fn),  "fn_rate": len(fn) / n_true,
        "fp":          len(fp),  "fp_rate": len(fp) / n_true,
        "ws":          ws,        "ws_rate": ws / n_true,
        "fn_barcodes": fn,
        "fp_barcodes": fp,
        "matched":     matched,
    }

**Note on `rapidfuzz` cutoff semantics:** with `score_cutoff = max_dist`, any pair whose true Levenshtein distance exceeds `max_dist` is returned as `max_dist + 1`. Pairs at or below the cutoff are returned with their real distance. So `min(row) <= max_dist` correctly flags an FP centroid whose nearest true barcode is within the WS threshold.

In [ ]:
print("Evaluating Shepherd...")
t0 = time.time()
shep_eval = evaluate(shepherd, true_counts)
print(f"  done in {time.time()-t0:.1f}s")

print("Evaluating barbac...")
t0 = time.time()
barbac_eval = evaluate(barbac, true_counts)
print(f"  done in {time.time()-t0:.1f}s")

## 5. Head-to-head comparison

In [ ]:
def _row(e):
    return [
        e["n_centroids"],
        round(e["pearson_r"], 4),
        f"{e['fn']:,} ({100*e['fn_rate']:.3f}%)",
        f"{e['fp']:,} ({100*e['fp_rate']:.3f}%)",
        f"{e['ws']:,} ({100*e['ws_rate']:.3f}%)",
    ]

comparison = pd.DataFrame(
    {"Shepherd": _row(shep_eval), "barbac": _row(barbac_eval)},
    index=["Centroids", "Pearson R (log10)", "FN", "FP", "WS"],
)
comparison

## 6. Where do they disagree?

In [ ]:
fn_shep = set(shep_eval["fn_barcodes"])
fn_barb = set(barbac_eval["fn_barcodes"])
fp_shep = set(shep_eval["fp_barcodes"])
fp_barb = set(barbac_eval["fp_barcodes"])

print("FALSE NEGATIVES (true barcodes missed)")
print(f"  Both miss:                 {len(fn_shep & fn_barb):,}")
print(f"  Shepherd catches, barbac misses: {len(fn_shep - fn_barb):,}")
print(f"  barbac catches, Shepherd misses: {len(fn_barb - fn_shep):,}")

print("\nFALSE POSITIVES (spurious centroids)")
print(f"  Both create:               {len(fp_shep & fp_barb):,}")
print(f"  Only Shepherd creates:     {len(fp_shep - fp_barb):,}")
print(f"  Only barbac creates:       {len(fp_barb - fp_shep):,}")

In [ ]:
# Which true barcodes does Shepherd catch that barbac misses, and at what abundance?
shep_wins = sorted(fn_barb - fn_shep)
if shep_wins:
    df = pd.DataFrame({"barcode": shep_wins}).merge(
        true_counts, on="barcode", how="left"
    ).sort_values("true_count", ascending=False)
    print("Top 20 barcodes Shepherd recovers but barbac misses (by true abundance):")
    display(df.head(20))
    print("\nAbundance summary:")
    display(df["true_count"].describe())
else:
    print("barbac catches every true barcode Shepherd does.")

## 7. Abundance correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, (name, e) in zip(axes, [("Shepherd", shep_eval), ("barbac", barbac_eval)]):
    m = e["matched"]
    ax.loglog(m["true_count"], m["sum_counts"], ".", alpha=0.25, markersize=2)
    lim = [min(m["true_count"].min(), m["sum_counts"].min()),
           max(m["true_count"].max(), m["sum_counts"].max())]
    ax.plot(lim, lim, "k--", alpha=0.5, linewidth=1)
    ax.set_xlabel("True count")
    ax.set_ylabel("Inferred count")
    ax.set_title(f"{name}  (R = {e['pearson_r']:.4f}, n = {len(m):,})")
fig.tight_layout()
plt.show()

## 8. Save the comparison

Writes a CSV next to the data so this can be re-loaded without re-running anything.

In [ ]:
summary = pd.DataFrame({
    "method":      ["Shepherd", "barbac"],
    "n_centroids": [shep_eval["n_centroids"], barbac_eval["n_centroids"]],
    "pearson_r":   [shep_eval["pearson_r"],   barbac_eval["pearson_r"]],
    "fn":          [shep_eval["fn"],          barbac_eval["fn"]],
    "fn_rate":     [shep_eval["fn_rate"],     barbac_eval["fn_rate"]],
    "fp":          [shep_eval["fp"],          barbac_eval["fp"]],
    "fp_rate":     [shep_eval["fp_rate"],     barbac_eval["fp_rate"]],
    "ws":          [shep_eval["ws"],          barbac_eval["ws"]],
    "ws_rate":     [shep_eval["ws_rate"],     barbac_eval["ws_rate"]],
})
out_path = WORK / "benchmark_comparison.csv"
summary.to_csv(out_path, index=False)
print(f"Wrote {out_path}")
summary

## 9. Diagnostic deep dive — *why* are we missing or mis-calling barcodes?

Sections 1–8 give *what* the FN/FP/WS counts are. The cells below ask **where each miss went** and **how damaging each false positive is**, ported from the original R diagnostic scripts.

To run them we need per-cluster membership for both methods:

- **barbac** → `barbac_clusters.csv` (written by `export_barbac_result.R`)
- **Shepherd** → `shepherd_input_seq_clust.csv` (Shepherd writes this automatically) joined with the input counts and the centroid table.

In [ ]:
barbac_members = pd.read_csv(WORK / "barbac_clusters.csv")
print(f"barbac members: {len(barbac_members):,}  (clusters: {barbac_members['central_barcode'].nunique():,})")
barbac_members.head()

In [ ]:
# Shepherd's seq_clust.csv labels each input sequence with an integer cluster id.
# The id is the row index of the centroid in the *input file*, not the row index
# in pb_freq.csv. So we build the cluster_id -> centroid map by looking up each
# pb_freq centroid in seq_clust and taking its own cluster id as the key.
shep_clust = pd.read_csv(WORK / "shepherd_input_seq_clust.csv")
cent_self = shep_clust[shep_clust["sequence"].isin(shepherd["central_barcode"])]
shep_centroid_map = dict(zip(cent_self["cluster"], cent_self["sequence"]))
assert len(shep_centroid_map) == len(shepherd), \
    f"Mapping built {len(shep_centroid_map)} but expected {len(shepherd)} centroids"

shep_clust["central_barcode"] = shep_clust["cluster"].map(shep_centroid_map)
input_counts = input_data.set_index("barcode")["counts"]
shep_members = (
    shep_clust.rename(columns={"sequence": "member"})
    .assign(member_count=lambda d: d["member"].map(input_counts).fillna(0).astype(int))
    [["central_barcode", "member", "member_count"]]
    .dropna(subset=["central_barcode"])
)
print(f"Shepherd members: {len(shep_members):,}  (clusters: {shep_members['central_barcode'].nunique():,})")
shep_members.head()

### 9a. FN absorption — which cluster ate each missed barcode?

For every true barcode that the method *didn't* recover as a centroid, find the cluster whose member list contains it. Then look at:

- distance from FN → absorbing centroid (should be small if the merge was reasonable)
- whether the absorbing centroid is itself a true barcode (good — just a slightly off true call) or spurious (bad — a chimera)
- count ratio `centroid_count / fn_count` (low ratio = two true barcodes at similar abundance got collapsed — the most damaging kind of miss)
- how many FNs were never sequenced at all (count = 0 in input — unrecoverable, not the algorithm's fault)

In [ ]:
def fn_absorption(method_name, eval_result, members_df, true_counts, input_data):
    fn_set = set(eval_result["fn_barcodes"])
    true_set = set(true_counts["barcode"])
    true_count_map = true_counts.set_index("barcode")["true_count"]
    input_count_map = input_data.set_index("barcode")["counts"]

    fn_members = members_df[members_df["member"].isin(fn_set)].copy()
    fn_members["fn_count"] = fn_members["member"].map(true_count_map)
    fn_members["centroid_count"] = fn_members["central_barcode"].map(input_count_map)
    fn_members["distance"] = [
        Levenshtein.distance(a, b) for a, b in zip(fn_members["member"], fn_members["central_barcode"])
    ]
    fn_members["is_true_centroid"] = fn_members["central_barcode"].isin(true_set)
    fn_members["count_ratio"] = fn_members["centroid_count"] / fn_members["fn_count"]

    # FNs that aren't anywhere in the cluster output = never sequenced
    absorbed = set(fn_members["member"])
    never_sequenced = [b for b in fn_set if b not in absorbed and input_count_map.get(b, 0) == 0]
    only_missing_from_clusters = [b for b in fn_set if b not in absorbed and input_count_map.get(b, 0) > 0]

    print(f"=== {method_name} FN ABSORPTION ===")
    print(f"Total FN: {len(fn_set):,}")
    print(f"  absorbed by some cluster: {len(absorbed):,}")
    print(f"  never sequenced (count=0 in input): {len(never_sequenced):,}")
    print(f"  sequenced but dropped pre-clustering: {len(only_missing_from_clusters):,}")

    print("\nDistance to absorbing centroid:")
    print(fn_members["distance"].value_counts().sort_index().to_string())

    print("\nIs the absorbing centroid a true barcode?")
    print(fn_members["is_true_centroid"].value_counts().to_string())

    print("\nCount ratio (centroid / FN) by distance:")
    print(
        fn_members.groupby("distance")["count_ratio"]
        .agg(["size", "min", "median", "max"])
        .round(2)
        .to_string()
    )

    n_close_ratio = int((fn_members["count_ratio"] < 2).sum())
    n_high_count  = int((fn_members["fn_count"] > 100).sum())
    print(f"\nFN with count_ratio < 2 (two true barcodes at similar abundance): {n_close_ratio:,}")
    print(f"FN with true_count > 100 (high-abundance barcodes we're missing): {n_high_count:,}")
    return fn_members

barbac_fn_abs = fn_absorption("barbac", barbac_eval, barbac_members, true_counts, input_data)

In [ ]:
shep_fn_abs = fn_absorption("Shepherd", shep_eval, shep_members, true_counts, input_data)

### 9b. Single-FN deep dive

Pick one FN barcode and inspect *everything* about its fate: the cluster that absorbed it, the absorber's full member list, distances, and whether anything in the input was within d=1 of the missed barcode.

In [ ]:
def inspect_fn(fn_bc, method_name, members_df, true_counts, input_data, max_dist=MAX_DIST):
    true_set = set(true_counts["barcode"])
    true_count_map = true_counts.set_index("barcode")["true_count"]
    input_count_map = input_data.set_index("barcode")["counts"]

    print(f"=== {method_name}: deep dive on FN '{fn_bc}' ===")
    in_input = fn_bc in input_count_map.index
    print(f"In input?         {in_input}")
    print(f"True count:       {true_count_map.get(fn_bc, 'N/A')}")
    print(f"Input count:      {input_count_map.get(fn_bc, 0)}")

    abs_row = members_df[members_df["member"] == fn_bc]
    if abs_row.empty:
        print("\nFN is not present in any cluster output (likely never sequenced).")
        return

    abs_centroid = abs_row["central_barcode"].iloc[0]
    abs_cluster  = members_df[members_df["central_barcode"] == abs_centroid]
    dist_fn_to_abs = Levenshtein.distance(fn_bc, abs_centroid)

    print(f"\nAbsorbed by centroid:  {abs_centroid}")
    print(f"  distance FN -> centroid: {dist_fn_to_abs}")
    print(f"  centroid input count:    {input_count_map.get(abs_centroid, 'N/A')}")
    print(f"  centroid is true barcode? {abs_centroid in true_set}")
    print(f"  cluster size:            {len(abs_cluster):,} members")

    print("\nTop 10 members of absorbing cluster (by count):")
    print(abs_cluster.sort_values("member_count", ascending=False).head(10).to_string(index=False))

    nearest_dists = process.cdist(
        [abs_centroid], true_counts["barcode"].tolist(),
        scorer=Levenshtein.distance, score_cutoff=max_dist, dtype=np.uint8,
    )[0]
    nearest_idx = int(np.argmin(nearest_dists))
    nearest_true = true_counts["barcode"].iloc[nearest_idx]
    nearest_d    = int(nearest_dists[nearest_idx])
    print(f"\nNearest true barcode to absorbing centroid: {nearest_true}  (d={nearest_d})")

    # How many input sequences within d=1 of the FN?
    if in_input:
        sample_inputs = input_data["barcode"].sample(min(50_000, len(input_data)), random_state=0).tolist()
        close = [b for b in sample_inputs if Levenshtein.distance(b, fn_bc) <= 1]
        print(f"\nInput sequences within d=1 of FN (sampled 50k): {len(close)}")

# Example: inspect the highest-abundance FN (worst miss)
if barbac_eval["fn"] > 0:
    worst_fn = (
        true_counts[true_counts["barcode"].isin(barbac_eval["fn_barcodes"])]
        .sort_values("true_count", ascending=False)
        .iloc[0]["barcode"]
    )
    inspect_fn(worst_fn, "barbac", barbac_members, true_counts, input_data)
else:
    print("barbac has no false negatives to inspect.")

### 9c. FP / WS analysis — how damaging are the spurious centroids?

For every FP centroid: count it was assigned in the result, raw input count, nearest true barcode and distance, and whether it qualifies as a WS (Wrong Sequence — within `max_dist` of *some* true barcode, which is much worse than an orphan FP that's far from anything real).

In [ ]:
def fp_analysis(method_name, eval_result, members_df, true_counts, input_data, max_dist=MAX_DIST):
    fp_list = eval_result["fp_barcodes"]
    if not fp_list:
        print(f"{method_name} has no false positives.")
        return None

    true_list = true_counts["barcode"].tolist()
    true_count_arr = true_counts["true_count"].to_numpy()
    input_count_map = input_data.set_index("barcode")["counts"]
    cluster_count_map = (
        members_df.groupby("central_barcode")["member_count"].sum().to_dict()
    )

    rows = []
    batch = 200
    for i in range(0, len(fp_list), batch):
        chunk = fp_list[i : i + batch]
        dm = process.cdist(
            chunk, true_list,
            scorer=Levenshtein.distance, score_cutoff=max_dist, dtype=np.uint8,
        )
        nearest_idx = dm.argmin(axis=1)
        nearest_d   = dm[np.arange(len(chunk)), nearest_idx]
        for fp_bc, idx, d in zip(chunk, nearest_idx, nearest_d):
            nearest_true = true_list[idx]
            rows.append({
                "fp_barcode":        fp_bc,
                "fp_cluster_count":  cluster_count_map.get(fp_bc, 0),
                "fp_raw_count":      input_count_map.get(fp_bc, np.nan),
                "in_input":          fp_bc in input_count_map.index,
                "nearest_true":      nearest_true,
                "nearest_true_count": int(true_count_arr[idx]),
                "dist_to_true":      int(d),
                "is_ws":             int(d) <= max_dist,
            })
    df = pd.DataFrame(rows)
    df["count_ratio"] = df["nearest_true_count"] / df["fp_cluster_count"].replace(0, np.nan)

    print(f"=== {method_name} FP / WS ANALYSIS ===")
    print(f"Total FP: {len(df):,}   WS (d <= {max_dist}): {int(df['is_ws'].sum()):,}")
    print("\nIn input file?\n" + df["in_input"].value_counts().to_string())
    print("\nDistance to nearest true barcode:\n" + df["dist_to_true"].value_counts().sort_index().to_string())
    print("\nFP cluster count (after absorbing errors):")
    print(df["fp_cluster_count"].describe().round(1).to_string())
    print("\nTop 20 most damaging FP (highest cluster count):")
    print(df.sort_values("fp_cluster_count", ascending=False).head(20).to_string(index=False))
    print("\nTop 20 WS only (FP that look like real barcodes):")
    print(
        df[df["is_ws"]].sort_values(["dist_to_true", "fp_cluster_count"], ascending=[True, False])
        .head(20).to_string(index=False)
    )
    return df

barbac_fp_df = fp_analysis("barbac", barbac_eval, barbac_members, true_counts, input_data)

In [ ]:
shep_fp_df = fp_analysis("Shepherd", shep_eval, shep_members, true_counts, input_data)